# Social Text Analytics — Climate Change Reddit Comments
**Analyses:** Keyword Frequency · TF-IDF · N-Gram Context · Co-occurrence Network · VADER Sentiment · Controversiality · LDA Topic Modeling


## 0. Setup

In [30]:
import os, re, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

import nltk
import spacy
from scipy.stats import kruskal
from nltk.corpus import stopwords
from nltk.util import ngrams
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import networkx as nx

warnings.filterwarnings("ignore")


In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = r"C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\DATASETS\\"
OUT_DIR   = r"C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\"
os.makedirs(OUT_DIR, exist_ok=True)


In [3]:
# ── Stopwords (single consolidated definition) ─────────────────────────────────
_nltk_stops = set(stopwords.words("english"))

_custom_stops = {
    # Reddit artifacts
    "amp", "gt", "lt", "www", "http", "https", "reddit", "subreddit", "comment", "post",
    # Contractions & fragments
    "s", "t", "re", "ve", "ll", "don", "doesn", "isn", "won",
    # Generic high-frequency words that add no topic signal
    "like", "just", "people", "know", "think", "really", "good", "bad", "time",
    "see", "make", "want", "need", "lot", "things", "thing", "years", "year",
    "would", "could", "should", "also", "even", "much", "many", "get", "got",
    "one", "any", "most", "other", "still", "way", "may", "might", "must",
    "said", "say", "says", "well",
    # Year tokens
    "2024", "2025", "2026",
}

STOPWORDS = _nltk_stops | _custom_stops


In [ ]:
# ── Subreddits of interest ─────────────────────────────────────────────────────
FOCUS_SUBREDDITS = [
    "conspiracy", "changemyview", "climatechange", "energy", "climate",
    "Futurology", "politics", "europe", "worldnews", "science",
    "ClimateShitposting", "canada", "unitedkingdom", "news", "climateskeptics",
]


In [5]:
# ── Helper functions ───────────────────────────────────────────────────────────
def clean_text(text):
    """Strip URLs, keep only ASCII letters, lowercase."""
    if not isinstance(text, str):
        return ""
    #text = html.unescape(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return text.lower().strip()


def tokenize(text):
    """Clean → split → remove stopwords & short tokens."""
    return [w for w in clean_text(text).split() if w not in STOPWORDS and len(w) > 2]


def get_valid_ngrams(token_list, n=2):
    """Generate n-grams, filtering out phrases with single-char tokens."""
    valid = []
    for gram in ngrams(token_list, n):
        if any(len(w.strip()) <= 1 for w in gram if w.isalnum()):
            continue
        valid.append(" ".join(gram))
    return valid


def save_fig(fig, name):
    """Save figure to OUT_DIR and close."""
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


In [6]:
# ── spaCy lemmatizer (parser & NER disabled for speed) ─────────────────────────
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize(text):
    """Return list of lemmas after spaCy processing, filtered by STOPWORDS."""
    if not isinstance(text, str):
        return []
    return [
        token.lemma_.strip()
        for token in nlp(text.lower())
        if not (token.is_punct or token.is_space or token.is_stop)
        and len(token.lemma_.strip()) > 1
        and token.lemma_.strip() not in STOPWORDS
    ]


## 1. Load Data

In [7]:
print("Loading data …")
df = pd.read_csv(
    DATA_PATH + "cleaned_climate_comments_2025_05_to_2026_04.xls",
    usecols=["comment_id", "score", "self_text", "subreddit",
             "controversiality", "comment_month", "comment_word_count", "post_title"],
)
df["self_text"]       = df["self_text"].fillna("")
df["controversial"]   = df["controversiality"].astype(int)
df["comment_month"]   = pd.to_datetime(df["comment_month"], errors="coerce")
print(f"  {len(df):,} rows · {df['subreddit'].nunique()} subreddits")
df.head(3)


Loading data …
  569,506 rows · 24 subreddits


,comment_id,score,self_text,subreddit,controversiality,post_title,comment_month,comment_word_count,controversial
0,oj8793w,3,&gt; Sorry baby boy.\n\nI sense some hatred he...,climateskeptics,0,The Evidence Shows We Are Nowhere Near a Clima...,2026-04-01,14,0
1,oj877ch,2,WELL HACKSHUALLY! I don't think the trees were...,climatechange,0,Is Planting trees actually as effective agains...,2026-04-01,25,0
2,oj86mg5,1,"You said I'm wrong, but I'm not.\n\nSolar bare...",climatechange,0,China’s vast nuclear power sector is now able ...,2026-04-01,31,0


## 2. Tokenization & Sampling

In [8]:
SAMPLE = 80_000
#sample_df = df.sample(SAMPLE, random_state=42).copy()

sample_df = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 5000), random_state=42))
    .reset_index(drop=True)
)

print(f"Working sample: {SAMPLE:,} rows")

# Simple tokenization for frequency & TF-IDF tasks
print("  Tokenizing (simple) …")
sample_df["tokens"] = sample_df["self_text"].apply(tokenize)

# Lemmatized tokens for richer NLP tasks (LDA, co-occurrence)
print("  Lemmatizing (spaCy) …")
sample_df["lemma_tokens"] = sample_df["self_text"].apply(lemmatize)

# Controversial / non-controversial subsets from FULL df (30K each)
cont1 = df[df["controversial"] == 1].sample(min(15_000, df["controversial"].sum()), random_state=1)
cont0 = df[df["controversial"] == 0].sample(15_000, random_state=2)
cont1 = cont1.copy(); cont1["tokens"] = cont1["self_text"].apply(tokenize)
cont0 = cont0.copy(); cont0["tokens"] = cont0["self_text"].apply(tokenize)
print("Done.")


Working sample: 80,000 rows
  Tokenizing (simple) …
  Lemmatizing (spaCy) …
Done.


## 3. Keyword Frequency

In [9]:
print("[3] Global keyword frequency …")
all_tokens = [t for ts in sample_df["tokens"] for t in ts]
freq       = Counter(all_tokens)
top_n      = freq.most_common(30)

words, counts = zip(*top_n)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(words[::-1], counts[::-1], color=sns.color_palette("Blues_d", len(top_n)))
ax.set_xlabel("Frequency", fontsize=11)
ax.set_title("Top 30 Keywords — All Subreddits (80K sample)", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()
save_fig(fig, "01_keyword_frequency_global.png")


[3] Global keyword frequency …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\01_keyword_frequency_global.png


In [10]:
# Per-subreddit keyword heatmap (normalised by total vocabulary per subreddit)
print("[3] Per-subreddit keyword heatmap …")
sub_df = sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]

sub_freqs, sub_total_counts = {}, {}
for sub, grp in sub_df.groupby("subreddit"):
    toks = [t for ts in grp["tokens"] for t in ts]
    sub_freqs[sub]        = Counter(toks)
    sub_total_counts[sub] = len(toks)

sub_all      = Counter()
for c in sub_freqs.values(): sub_all.update(c)
top_sub_words = [w for w, _ in sub_all.most_common(20)]

heat_data = pd.DataFrame(
    {sub: [sub_freqs[sub].get(w, 0) for w in top_sub_words] for sub in sub_freqs},
    index=top_sub_words,
).T
sub_totals = pd.Series(sub_total_counts).reindex(heat_data.index).replace(0, 1)
heat_norm  = heat_data.div(sub_totals, axis=0) * 1000   # per-mille

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heat_norm, cmap="YlOrRd", linewidths=0.3, ax=ax,
            annot=True, fmt=".1f",
            cbar_kws={"label": "Relative freq per 1,000 tokens (‰)"})
ax.set_title("Keyword Relative Frequency per Subreddit (top 20 terms)", fontsize=13, fontweight="bold")
ax.set_xlabel("Keyword"); ax.set_ylabel("Subreddit")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()
save_fig(fig, "02_keyword_heatmap_subreddits.png")


[3] Per-subreddit keyword heatmap …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\02_keyword_heatmap_subreddits.png


## 4. TF-IDF per Subreddit

In [11]:
print("[4] TF-IDF per subreddit …")
sub_docs = (
    sub_df.groupby("subreddit")["self_text"]
    .apply(lambda x: " ".join(x.dropna().apply(clean_text)))
    .reset_index()
)
sub_docs.columns = ["subreddit", "text_clean"]

vectorizer   = TfidfVectorizer(max_features=5000, stop_words=list(STOPWORDS),
                               ngram_range=(1, 2), min_df=2)
tfidf_matrix = vectorizer.fit_transform(sub_docs["text_clean"])
feature_names = vectorizer.get_feature_names_out()

tfidf_records = []
for i, sub in enumerate(sub_docs["subreddit"]):
    row     = tfidf_matrix[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:10]
    for rank, idx in enumerate(top_idx):
        tfidf_records.append({"subreddit": sub, "rank": rank + 1,
                              "term": feature_names[idx], "tfidf": round(float(row[idx]), 5)})

tfidf_df = pd.DataFrame(tfidf_records)
tfidf_df.to_csv(os.path.join(OUT_DIR, "03_tfidf_top10_per_subreddit.csv"), index=False)
print("  Saved → 03_tfidf_top10_per_subreddit.csv")

# Facet plot
plot_subs = [s for s in FOCUS_SUBREDDITS if s in sub_docs["subreddit"].values][:12]
n_cols = 4
n_rows = int(np.ceil(len(plot_subs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.2))
axes = axes.flatten()
palette = sns.color_palette("tab20", 10)

for ax, sub in zip(axes, plot_subs):
    sub_data = tfidf_df[tfidf_df["subreddit"] == sub].head(8)
    if sub_data.empty: ax.set_visible(False); continue
    ax.barh(sub_data["term"][::-1].values, sub_data["tfidf"][::-1].values, color=palette)
    ax.set_title(f"r/{sub}", fontsize=9, fontweight="bold")
    ax.tick_params(axis="y", labelsize=7); ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(plot_subs):]: ax.set_visible(False)
fig.suptitle("Top TF-IDF Terms per Subreddit (unigrams + bigrams)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
save_fig(fig, "03_tfidf_facets.png")


[4] TF-IDF per subreddit …
  Saved → 03_tfidf_top10_per_subreddit.csv
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\03_tfidf_facets.png


## 5. N-Gram Context Discovery

In [12]:
print("[5] N-gram context discovery …")
N_GRAM_SIZE = 2  # change to 3 for trigrams

all_phrases = [p for ts in sample_df["tokens"] for p in get_valid_ngrams(ts, n=N_GRAM_SIZE)]
top_phrases = Counter(all_phrases).most_common(25)

if top_phrases:
    phrases, counts = zip(*top_phrases)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(phrases[::-1], counts[::-1],
            color=plt.cm.viridis_r(np.linspace(0, 1, len(top_phrases))))
    ax.set_xlabel("Phrase Count", fontsize=11)
    ax.set_ylabel("Phrase", fontsize=11)
    ax.set_title(f"Top {len(top_phrases)} Multi-Word Phrases (N={N_GRAM_SIZE})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    save_fig(fig, f"04_context_ngrams_n{N_GRAM_SIZE}.png")


[5] N-gram context discovery …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\04_context_ngrams_n2.png


## 6. Keyword Co-occurrence Network

In [13]:
print("[6] Building co-occurrence network …")
MAX_NODES  = 25
MIN_WEIGHT = 5   # raise if graph is too dense (e.g. 50)

all_tokens_flat = [t for ts in sample_df["tokens"] for t in ts]
top_words = {w for w, _ in Counter(all_tokens_flat).most_common(MAX_NODES)}
global_counts = Counter(all_tokens_flat)

co_counts = Counter()
for tokens in sample_df["tokens"]:
    valid = sorted({t for t in tokens if t in top_words})
    if len(valid) >= 2:
        for w1, w2 in itertools.combinations(valid, 2):
            co_counts[(w1, w2)] += 1

G = nx.Graph()
for (w1, w2), weight in co_counts.items():
    if weight > MIN_WEIGHT:
        G.add_edge(w1, w2, weight=weight)

fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.4, iterations=50, seed=42)

node_sizes  = [global_counts[n] * 0.15 for n in G.nodes()]
edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_widths = [1 + (w / max_w) * 5 for w in edge_weights]

nx.draw_networkx_edges(G, pos, ax=ax, width=edge_widths, edge_color="skyblue", alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                       node_color=sns.color_palette("viridis", len(G.nodes())), alpha=0.85)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=10, font_weight="bold",
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2))
ax.set_title("Keyword Co-occurrence Network", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()
save_fig(fig, "05_cooccurrence_network.png")


[6] Building co-occurrence network …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\05_cooccurrence_network.png


## 7. VADER Sentiment Analysis

In [14]:
print("[7] VADER sentiment scoring …")
try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    nltk.download("vader_lexicon")

analyzer = SentimentIntensityAnalyzer()
scores = sample_df["self_text"].apply(
    lambda t: analyzer.polarity_scores(t if isinstance(t, str) else "")
)
sample_df["vader_neg"]      = scores.apply(lambda s: s["neg"])
sample_df["vader_pos"]      = scores.apply(lambda s: s["pos"])
sample_df["vader_compound"] = scores.apply(lambda s: s["compound"])
sample_df["sentiment_label"] = pd.cut(
    sample_df["vader_compound"],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=["Negative", "Neutral", "Positive"],
)
print("  Done.")


[7] VADER sentiment scoring …
  Done.


In [15]:
# 7a. Global distribution (histogram + pie)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sample_df["vader_compound"], bins=60,
             color="#4C72B0", edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.2, label="Neutral boundary")
axes[0].set_xlabel("VADER Compound Score", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_title("Distribution of VADER Compound Scores", fontsize=12, fontweight="bold")
axes[0].legend()

label_counts = sample_df["sentiment_label"].value_counts()
axes[1].pie(label_counts, labels=label_counts.index, autopct="%1.1f%%",
            colors=["#d62728", "#aec7e8", "#2ca02c"], startangle=140,
            textprops={"fontsize": 11})
axes[1].set_title("Sentiment Label Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
save_fig(fig, "06a_vader_global_distribution.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\06a_vader_global_distribution.png


In [16]:
# 7b. Mean sentiment by subreddit
sub_sent = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")["vader_compound"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
    .rename(columns={"mean": "mean_compound", "median": "median_compound",
                     "std": "std_compound", "count": "n_comments"})
    .sort_values("mean_compound")
)
sub_sent.to_csv(os.path.join(OUT_DIR, "06_sentiment_by_subreddit.csv"), index=False)

fig, ax = plt.subplots(figsize=(11, 7))
colors = ["#d62728" if v < 0 else "#2ca02c" for v in sub_sent["mean_compound"]]
ax.barh(sub_sent["subreddit"], sub_sent["mean_compound"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Mean VADER Compound Score", fontsize=11)
ax.set_title("Mean Sentiment by Subreddit", fontsize=13, fontweight="bold")
ax.set_xlim(-0.25, 0.25)
plt.tight_layout()
save_fig(fig, "06b_sentiment_by_subreddit.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\06b_sentiment_by_subreddit.png


In [20]:
# 7b-2. Statistical test: is the sentiment difference across subreddits significant?
# Kruskal-Wallis is a non-parametric alternative to one-way ANOVA — it does not
# assume normality, which fits compound scores bounded in [-1, 1].
print("[7] Kruskal-Wallis test on sentiment across subreddits …")
sentiment_groups = [
    grp["vader_compound"].values
    for _, grp in sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)].groupby("subreddit")
]
kw_stat, kw_p = kruskal(*sentiment_groups)
print(f"  H-statistic = {kw_stat:.2f}, p-value = {kw_p:.4g}")
if kw_p < 0.05:
    print("  → Sentiment distributions differ significantly across subreddits (p < 0.05).")
else:
    print("  → No statistically significant difference in sentiment across subreddits (p >= 0.05).")


[7] Kruskal-Wallis test on sentiment across subreddits …
  H-statistic = 568.51, p-value = 2.648e-112
  → Sentiment distributions differ significantly across subreddits (p < 0.05).


In [21]:
# 7c. Sentiment over time
monthly = (
    sample_df.dropna(subset=["comment_month"])
    .groupby("comment_month")["vader_compound"]
    .agg(["mean", "median"]).reset_index()
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["comment_month"], monthly["mean"],   marker="o", label="Mean",   color="#4C72B0")
ax.plot(monthly["comment_month"], monthly["median"], marker="s", linestyle="--", label="Median", color="#DD8452")
ax.axhline(0, color="grey", linewidth=0.7, linestyle=":")
ax.set_xlabel("Month", fontsize=11); ax.set_ylabel("VADER Compound Score", fontsize=11)
ax.set_title("Sentiment Trend Over Time", fontsize=13, fontweight="bold"); ax.legend()
plt.xticks(rotation=30); plt.tight_layout()
save_fig(fig, "06c_sentiment_trend_monthly.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\06c_sentiment_trend_monthly.png


In [22]:
# 7d. Stacked sentiment composition per subreddit
sub_label = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby(["subreddit", "sentiment_label"]).size().unstack(fill_value=0)
)
sub_label_pct = sub_label.div(sub_label.sum(axis=1), axis=0) * 100
sub_label_pct = sub_label_pct.reindex(columns=["Negative", "Neutral", "Positive"])
sub_label_pct = sub_label_pct.sort_values("Negative", ascending=False)

fig, ax = plt.subplots(figsize=(13, 7))
sub_label_pct.plot(kind="barh", stacked=True, ax=ax, width=0.6,
                   color={"Negative": "#e74c3c", "Neutral": "#95a5a6", "Positive": "#2ecc71"})
ax.set_xlabel("% of comments", fontsize=11)
ax.set_title("Sentiment Composition per Subreddit", fontsize=13, fontweight="bold")
ax.legend(title="Sentiment", bbox_to_anchor=(1.05, 1), loc="upper left")
for p in ax.patches:
    width = p.get_width()
    if width > 5:
        ax.text(p.get_x() + width / 2, p.get_y() + p.get_height() / 2,
                f"{width:.1f}%", ha="center", va="center",
                color="white", fontweight="bold", fontsize=9)
plt.tight_layout()
save_fig(fig, "06d_sentiment_stacked_subreddits.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\06d_sentiment_stacked_subreddits.png


## 8. Sentiment-Filtered Context Discovery

In [23]:
# ── Subreddits of interest ─────────────────────────────────────────────────────
FOCUS_SUBREDDITS = [
    "conspiracy", "changemyview", "climatechange", "energy", "climate",
    "Futurology", "politics", "europe", "worldnews", "science",
    "ClimateShitposting", "canada", "unitedkingdom", "news", "climateskeptics",
]

In [24]:
print("[8] Sentimental context discovery …")
TARGET_SUB = "worldnews"   # change to any subreddit in FOCUS_SUBREDDITS

sub_focused = sample_df[sample_df["subreddit"] == TARGET_SUB]
neg_phrases = [p for ts in sub_focused[sub_focused["sentiment_label"] == "Negative"]["tokens"]
               for p in get_valid_ngrams(ts, n=2)]
pos_phrases = [p for ts in sub_focused[sub_focused["sentiment_label"] == "Positive"]["tokens"]
               for p in get_valid_ngrams(ts, n=2)]

top_neg = Counter(neg_phrases).most_common(15)
top_pos = Counter(pos_phrases).most_common(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
if top_neg:
    phrases_n, counts_n = zip(*top_neg)
    ax1.barh(phrases_n[::-1], counts_n[::-1], color=sns.color_palette("Reds_r", len(top_neg)))
    ax1.set_title(f"Negative Context — r/{TARGET_SUB}", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Phrase Count")
if top_pos:
    phrases_p, counts_p = zip(*top_pos)
    ax2.barh(phrases_p[::-1], counts_p[::-1], color=sns.color_palette("YlGn_r", len(top_pos)))
    ax2.set_title(f"Positive Context — r/{TARGET_SUB}", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Phrase Count")
plt.suptitle(f"Thematic Contrast — r/{TARGET_SUB}", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()
save_fig(fig, f"07_sentimental_context_{TARGET_SUB}.png")


[8] Sentimental context discovery …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\07_sentimental_context_worldnews.png


## 9. Controversiality Analysis

In [25]:
print("[9] Controversiality rate by subreddit …")
cont_rate = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")["controversial"]
    .agg(["mean", "sum", "count"]).reset_index()
    .rename(columns={"mean": "controversy_rate", "sum": "n_controversial", "count": "n_total"})
    .sort_values("controversy_rate", ascending=False)
)
cont_rate.to_csv(os.path.join(OUT_DIR, "08_controversiality_by_subreddit.csv"), index=False)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(cont_rate["subreddit"][::-1], cont_rate["controversy_rate"][::-1] * 100,
        color=sns.color_palette("Reds_r", len(cont_rate)))
ax.set_xlabel("Controversy Rate (%)", fontsize=11)
ax.set_title("Controversiality Rate by Subreddit", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
save_fig(fig, "08a_controversiality_by_subreddit.png")


[9] Controversiality rate by subreddit …
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\08a_controversiality_by_subreddit.png


In [26]:
# Sentiment vs. controversiality (grouped bar)
cont_sent = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby(["subreddit", "controversial"])["vader_compound"]
    .mean().unstack().reset_index()
)
cont_sent.columns = ["subreddit", "not_controversial", "controversial"]
cont_sent = cont_sent.dropna().sort_values("not_controversial")

x = np.arange(len(cont_sent)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, cont_sent["not_controversial"], w, label="Not controversial", color="#4C72B0")
ax.bar(x + w/2, cont_sent["controversial"],     w, label="Controversial",     color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels(cont_sent["subreddit"], rotation=40, ha="right", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Mean VADER Compound Score", fontsize=11)
ax.set_title("Sentiment: Controversial vs. Non-Controversial Comments", fontsize=12, fontweight="bold")
ax.legend(); plt.tight_layout()
save_fig(fig, "08b_sentiment_vs_controversiality.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\08b_sentiment_vs_controversiality.png


In [27]:
# Controversiality over time (dual-axis)
cont_monthly = (
    df.dropna(subset=["comment_month"])
    .groupby("comment_month")["controversial"]
    .agg(["mean", "sum", "count"]).reset_index()
)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(cont_monthly["comment_month"], cont_monthly["sum"], width=20,
        color="#d62728", alpha=0.5, label="# Controversial comments")
ax1.set_ylabel("Count", color="#d62728", fontsize=11)
ax1.tick_params(axis="y", labelcolor="#d62728")
ax2 = ax1.twinx()
ax2.plot(cont_monthly["comment_month"], cont_monthly["mean"] * 100,
         color="#333333", marker="o", linewidth=1.5, label="Rate (%)")
ax2.set_ylabel("Controversy Rate (%)", fontsize=11)
ax1.set_title("Controversiality Over Time", fontsize=13, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.xticks(rotation=30); plt.tight_layout()
save_fig(fig, "08c_controversiality_trend.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\08c_controversiality_trend.png


In [28]:
# Top keywords: controversial vs. non-controversial
for label_val, grp_df, tag, color in [
    (1, cont1, "controversial",     "#d62728"),
    (0, cont0, "non_controversial", "#4C72B0"),
]:
    toks  = [t for ts in grp_df["tokens"] for t in ts]
    top20 = Counter(toks).most_common(20)
    words_l, counts_l = zip(*top20)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(words_l[::-1], counts_l[::-1], color=color)
    ax.set_title(
        f"Top Keywords — {'Controversial' if label_val == 1 else 'Non-Controversial'} Comments",
        fontsize=12, fontweight="bold",
    )
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()
    save_fig(fig, f"09_keywords_{tag}.png")


  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\09_keywords_controversial.png
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\09_keywords_non_controversial.png


## 10. LDA Topic Modeling

In [29]:
print("[10] LDA topic modeling …")
N_TOPICS    = 5
N_TOP_WORDS = 10

corpus = sample_df["tokens"].apply(lambda ts: " ".join(str(t) for t in ts))
count_vec = CountVectorizer(max_df=0.95, min_df=2)
dtm = count_vec.fit_transform(corpus)

lda = LatentDirichletAllocation(
    n_components=N_TOPICS, max_iter=10,
    learning_method="online", random_state=42, n_jobs=-1,
)
lda.fit(dtm)
print("  LDA fitted.")

feature_names = count_vec.get_feature_names_out()
fig, axes = plt.subplots(1, N_TOPICS, figsize=(20, 6), sharex=False)
for idx, topic in enumerate(lda.components_):
    top_idx   = topic.argsort()[:-N_TOP_WORDS - 1:-1]
    top_words = [feature_names[i] for i in top_idx]
    top_wts   = [topic[i] for i in top_idx]
    ax = axes[idx]
    ax.barh(top_words[::-1], top_wts[::-1], color=sns.color_palette("muted")[idx % 6])
    ax.set_title(f"Topic #{idx + 1}", fontsize=12, fontweight="bold")
    ax.tick_params(labelsize=10)
    ax.grid(axis="x", linestyle="--", alpha=0.5)
plt.suptitle("LDA Topic Modeling — Top Terms per Topic", fontsize=15, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()
save_fig(fig, "10_lda_topics.png")


[10] LDA topic modeling …
  LDA fitted.
  Saved → C:\Users\Hungthq\MITB\ISSS606_ADS_in_Social_Networks\Group_Project\GIT\PROCESSED\\10_lda_topics.png


## 11. Summary Export

In [ ]:
print("[11] Writing subreddit summary …")
summary = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")
    .agg(
        n_comments       = ("comment_id",        "count"),
        mean_score       = ("score",             "mean"),
        median_score     = ("score",             "median"),
        n_controversial  = ("controversial",     "sum"),
        controversy_rate = ("controversial",     "mean"),
        mean_word_count  = ("comment_word_count","mean"),
    )
    .reset_index().round(4)
)
summary.to_csv(os.path.join(OUT_DIR, "00_subreddit_summary.csv"), index=False)
print("  Saved → 00_subreddit_summary.csv")
print("\n✓ All analyses complete.")
summary
